# 0 · Teach a Language Model What Sells Together

The plan for the whole drill: build an invented shop where I *know* which items sell together, get the exact answer with
classic itemset mining, then teach a small language model on the baskets and see how much of that answer it picked up.

This first notebook is just the warm-up: what an itemset and its support are, and why we can't count every combination.

## Support on six baskets

An **itemset** is a group of items. Its **support** is the fraction of baskets that contain all of them.

In [1]:
from itertools import combinations

baskets = [
    ["tea", "sugar", "milk"],
    ["tea", "sugar"],
    ["bread", "butter", "milk"],
    ["tea", "milk"],
    ["bread", "butter"],
    ["tea", "sugar", "bread"],
]

def support(itemset, baskets):
    """The share of baskets that hold every item in the itemset."""
    return sum(set(itemset) <= set(basket) for basket in baskets) / len(baskets)

for itemset in [("tea",), ("tea", "sugar"), ("bread", "butter"), ("tea", "butter")]:
    print(f"{' + '.join(itemset):16} support = {support(itemset, baskets):.2f}")

tea              support = 0.67
tea + sugar      support = 0.50
bread + butter   support = 0.33
tea + butter     support = 0.00


Tea + sugar shows up in 3 of 6 baskets → 0.50. Tea + butter never shows up together → 0.

## Frequent itemsets by brute force

Keep every itemset with support ≥ 0.5. With 5 items we can afford to try every combination.

In [2]:
items = sorted({item for basket in baskets for item in basket})
min_support = 0.5

for size in (1, 2, 3):
    for itemset in combinations(items, size):
        s = support(itemset, baskets)
        if s >= min_support:
            print(f"{' + '.join(itemset):16} {s:.2f}")

bread            0.50
milk             0.50
sugar            0.50
tea              0.67
sugar + tea      0.50


## Why brute force doesn't scale

A real shop has ~1,000 items. The number of possible itemsets explodes with size:

In [3]:
from math import comb

for size in (2, 3, 4, 5):
    print(f"itemsets of {size} items out of 1,000 items: {comb(1000, size):>20,}")

itemsets of 2 items out of 1,000 items:              499,500
itemsets of 3 items out of 1,000 items:          166,167,000
itemsets of 4 items out of 1,000 items:       41,417,124,750
itemsets of 5 items out of 1,000 items:    8,250,291,250,200


Trillions of 5-item sets — nobody counts those one at a time. **Apriori** (section 6) avoids it: a set can only be
frequent if all its subsets are, so it grows candidates level by level from frequent pieces only. It stays exact.

## The plan in one line each

1. Invent a shop (1,000 items, realistic sales) and plant 32 itemsets at known frequencies (5% down to 0.05%).
2. Mine the exact answer with Apriori.
3. Write baskets as text and continue pretraining a small open model on them (Phase 1 in miniature).
4. Probe the model and score how many planted itemsets it recovers at each frequency.

Invented brands matter: a model off the internet already "knows" tea goes with sugar, so made-up items and odd combos
let us tell what it learned *from the baskets*.

Sections 8–11 run on CPU PyTorch here (no Mac), so those timings and numbers will differ from the course.

## Test yourself

**1. Why is the support of tea+sugar never higher than the support of tea alone?**
Every basket that holds tea *and* sugar also holds tea. So the tea+sugar baskets are a subset of the tea baskets, and
their share can only be equal or smaller. Adding items to a set can never raise its support.

**2. If a 3-item set is frequent, what must be true of each of its 2-item subsets?**
Each of them must be frequent too, with support at least as high as the 3-item set. This is the rule Apriori uses:
only build bigger candidates out of pieces that are already frequent, and skip the rest.

**3. Why does the course use an invented shop instead of real sales data?**
Real data has no known right answer to check the model against. A model trained on the internet also already
"knows" that tea goes with sugar. Made-up items and deliberately planted combos mean anything the model gets right
must have come from our baskets.

## My notes

(Dakshitha to fill)